# Lecture 4 — Equilibrium and Aiyagari

**Computational Methods for Heterogeneous-Agent Macro**

Jeffrey Sun


#### The Aiyagari Model

Like many models, the Aiyagari model consists of three blocks:
1. A household block made of **Stages**
2. A firm side turning capital and labour into goods
3. A notion of equilibrium (steady state)

Familiar economics:
1. A demand side
2. A supply side
3. A notion of equilibrium


### Environment


In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using Plots, Random, LinearAlgebra

# Core Code


## The Household Block


### Parameters


In [ ]:
u(c) = c <= 0 ? -Inf : log(c)
loggrid(lo, hi, N) = exp.(range(log(lo), log(hi); length=N))

In [ ]:
function get_params(;
        β=0.96,
        R=1.04,
        b_grid=loggrid(0.05, 200.0, 400),
        z_grid=[2.7, 2.6, 2.5],
        P_z=[0.95 0.04 0.01;
           0.05 0.9  0.05;
           0.01 0.04 0.95],
    )

    return params = (;β, R,
                    V_shape=(length(b_grid), length(z_grid)),
                    b_grid, z_grid, P_z)
end

### Utility and Log Grid


In [ ]:
"""
Snap a value `x` to the nearest grid index.

"""
snap_idx(grid, x) = argmin(abs.(grid .- x))

### Backward iteration in stages

The three stages compose into one backward step on a 2-D value function
$V \in \mathbb{R}^{N_b \times N_z}$. Each stage maps one labelled $V$ to the next:

- **Stage 3 (Consumption-Saving) backward.** Grid-search $V^{\mathrm{end}} \mapsto V$.
- **Stage 2 (Income) backward.** Lookup $V \mapsto V^{\mathrm{start}}_{\mathrm{post}}$ at $R\,b^{\mathrm{end}} + z_{\mathrm{grid}}[j]$.
- **Stage 1 (Income Shock) backward.** Matrix-multiply $V^{\mathrm{start}}_{\mathrm{post}} \mapsto V^{\mathrm{start}}_{\mathrm{pre}}$ by $\Pi^\top$.

At the period boundary, $V^{\mathrm{end}} = \beta\, V^{\mathrm{start}}_{\mathrm{pre}}$.


In [ ]:
"""
Stage 1 (Income Shock) backward: V_start = V_pre_inc * P_z'.

"""
income_shock_backward(V_post_inc_shock, params) = V_post_inc_shock * params.P_z'

"""
Stage 2 (Income) backward: V_pre_inc(b_end, z) = V(R*b_end + z, z), snapped to nearest grid.
Fully broadcast — no explicit loop.

"""
function income_backward(V_post_inc, params)
    (;R, b_grid, z_grid) = params
    b_post_inc = snap_idx.(Ref(b_grid), R .* b_grid .+ z_grid')
    return V_pre_inc = V_post_inc[CartesianIndex.(b_post_inc, axes(b_post_inc, 2)')]
end

"""
Stage 3 (Consumption-Saving) backward: grid-search V_end → V over b_end choices.

"""
function consumption_saving_backward(V_end, params)
    # Unpack b_grid
    (;b_grid) = params
    
    # Move b_grid to dimension 3 to represent b_end choices
    b_next_grid = insertdims(b_grid; dims=(1,2))

    # Move b_grid part of V_end to dimension 3 to represent b_end choices
    # V_end_reshape shape is now [1, z_grid, b_grid]
    V_end_reshape = insertdims(permutedims(V_end, (2,1)); dims=1)

    # Maximize along b_end dimension (dim=3)
    V = maximum(u.(b_grid .- b_next_grid) .+ V_end_reshape; dims=3)

    # Drop dimension 3 from V and return
    return dropdims(V; dims=3)
end

"""
Read the optimal b_end policy off V_end, by argmax instead of max.

"""
function policy_function(V_end, params)
    # Unpack b_grid
    (;b_grid) = params
    
    # Move b_grid to dimension 3 to represent b_end choices
    b_next_grid = insertdims(b_grid; dims=(1,2))

    # Move b_grid part of V_end to dimension 3 to represent b_end choices
    # V_end_reshape shape is now [1, z_grid, b_grid]
    V_end_reshape = insertdims(permutedims(V_end, (2,1)); dims=1)

    # Maximize along b_end dimension (dim=3)
    policy_idxs = argmax(u.(b_grid .- b_next_grid) .+ V_end_reshape; dims=3)

    return [idx[3] for idx in policy_idxs[:,:,1]]
end

In [ ]:
function bellman_operator(V_end, params)
    (;β) = params
    # Stage 3
    V_post_inc = consumption_saving_backward(V_end, params)

    # Stage 2
    V_post_inc_shock = income_backward(V_post_inc, params)

    # Stage 1
    V_start = income_shock_backward(V_post_inc_shock, params)

    # Passage of time
    V_end_new = β .* V_start
    return V_end_new
end

### Value Function Iteration

Iterate $V \mapsto \mathcal{T}V$ until the max-abs change drops below `tol`.


In [ ]:
function solve_vfi(params; tol=1e-6, maxiter=2000, verbosity=0)
    V = zeros(params.V_shape)

    Δ, iters = Inf, 0
    while Δ >= tol
        TV = bellman_operator(V, params)
        Δ  = maximum(abs.(TV .- V))
        V  = TV
        iters += 1
        iters > maxiter && error("VFI did not converge in $maxiter iterations")
    end

    verbosity >= 1 && println("VFI converged in $iters iterations with error $Δ")
    return V
end

In [ ]:
params = get_params()
V = solve_vfi(params; verbosity=1)

### Forward iteration in stages

We can simulate each population matrix forward though each stage, one at a time.

- **Stage 1 (Income Shock) forward.** $\lambda \mapsto \lambda\, \Pi$.
- **Stage 2 (Income) forward.** Each $(b^{\mathrm{end}}, z)$ cell moves to $(R\,b^{\mathrm{end}} + z_{\mathrm{grid}}[j],\, z)$.
- **Stage 3 (Consumption-Saving) forward.** Each $(b, z)$ cell moves to $(b - c^\star(b, z),\, z)$.

Both wealth re-bins snap to the nearest grid point.


In [ ]:
"""
Stage 1 forward: λ_post = λ_pre * P_z.

"""
income_shock_forward(λ, P_z) = λ * P_z

"""
Stage 2 forward: each (i_b_end, i_z) cell moves to (snap(R*b_end + z), i_z).
Vectorize the destination index via broadcasting; one tight scatter.

"""

function change_λ_wealth(λ, b_inds_new)
    λ_new = zero(λ)
    for old_idx in CartesianIndices(λ)
        λ_new[b_inds_new[old_idx], old_idx[2]] += λ[old_idx]
    end
    return λ_new
end

function income_forward(λ, params)
    (;R, b_grid, z_grid) = params
    dest = snap_idx.(Ref(b_grid), R .* b_grid .+ z_grid')               # (N_b, N_z)
    return change_λ_wealth(λ, dest)
end

"""
Stage 3 forward: each (i_b, i_z) cell moves to (c_ind[i_b, i_z], i_z).
c_ind is exactly the destination index — direct scatter.

"""
consumption_saving_forward(λ, b_inds_new, params) = change_λ_wealth(λ, b_inds_new)

"""
One forward step of the composite operator T*:
λ → income shock → income → consumption saving.

"""
function simulate_population_forward(λ, b_inds_new, params)
    # Stage 1
    λ = income_shock_forward(λ, params.P_z)

    # Stage 2
    λ = income_forward(λ, params)

    # Stage 3
    λ = consumption_saving_forward(λ, b_inds_new, params)
    
    return λ
end


In [ ]:
function find_steady_state_population(V_end, params; tol=1e-5, maxiter=10_000, verbosity=0)

    b_inds_new = policy_function(V_end, params)
    
    λ = fill(1/length(params.V_shape), params.V_shape) # Start with households evenly distributed across gridpoints

    Δ, iters = Inf, 0
    while Δ >= tol
        λ_new = simulate_population_forward(λ, b_inds_new, params)
        Δ = maximum(abs.(λ_new .- λ))
        λ = λ_new
        iters += 1
        iters > maxiter && error("Steady state λ did not converge in $maxiter iterations")
    end

    verbosity >= 1 && println("Steady state λ converged in $iters iterations with error $Δ")
    return λ
end

In [ ]:
function solve_aiyagari_steady_state(params; verbosity=0)
    V_end = solve_vfi(params; verbosity)
    λ = find_steady_state_population(V_end, params; verbosity)
    return (;V_end, λ)
end

function solve_aiyagari_steady_state(;verbosity=0, param_vals...)
    return solve_aiyagari_steady_state(get_params(;param_vals...); verbosity)
end

In [ ]:
params = get_params(;R=1.04)
(;V_end, λ) = solve_aiyagari_steady_state(params; verbosity=0)

In [ ]:
sum(λ .* params.b_grid) # Aggregate capital in the economy

In [ ]:
params.b_grid

## Computing Moments


### Cross-sectional aggregates from $V$ and $c^\star$

Aggregate welfare $\bar V$, consumption $\bar C$, wealth $\bar K$ at each $t$ — all inner products against the marginal wealth distribution.


In [ ]:
get_c_policy(b_grid, b_grid_policy) = b_grid .- b_grid[b_grid_policy]

function compute_household_aggregates(;param_vals...)
    params = get_params(;param_vals...)
    (V_end, λ) = solve_aiyagari_steady_state(params)

    
    b_grid_policy = policy_function(V_end, params)
    c_policy = get_c_policy(params.b_grid, b_grid_policy)
    
    L_agg = sum(λ)
    K_agg = sum(params.b_grid .* λ)
    V_agg = sum(V_end .* λ)
    C_agg = sum(c_policy .* λ)

    return (;V_end, λ, L_agg, K_agg, V_agg, C_agg)
end

In [ ]:
params = get_params(;R=1.04)
(V_end, λ) = solve_aiyagari_steady_state(params)

@show V_agg = sum(V_end .* λ)
@show K_agg = sum(params.b_grid .* λ)

b_grid_policy = policy_function(V_end, params)
c_grid_policy = get_c_policy(params.b_grid, b_grid_policy)

@show C_agg = sum(c_grid_policy .* λ)

@show L_agg = sum(λ)


In [ ]:
compute_Y(K, L; α=1/3) = K^α * L^(1-α)

In [ ]:
res = compute_household_aggregates(;R=1.03)

In [ ]:
Y_agg = compute_Y(res.K_agg, res.L_agg)

In [ ]:
res.C_agg

In [ ]:
function compute_excess_demand(R)
    res = compute_household_aggregates(;R)
    
    Y_agg = compute_Y(res.K_agg, res.L_agg)

    return excess_demand = (res.C_agg - Y_agg)/Y_agg
end

In [ ]:
compute_excess_demand(1.05)

In [ ]:
tol = 0.01
lr = 0.001

R = 1.01
err = Inf
while err > tol
    excess_demand = compute_excess_demand(R)
    
    R_new = R - excess_demand*lr
    err = abs(excess_demand)

    R = R_new
    
    println("R=$R\t excess_demand=$excess_demand")
end

println("Equilibrium R: $R")